# 06: Portfolio and lockbox

## Commitments, written before any result

This notebook opens the lockbox (2025-07-01 to 2026-07-06) once. The three
sleeves (seasonality: weekday-halved momentum; orderflow: taker follow,
k=10; carry: short high funding, k=7) were selected on the dev window
(2020-01-01 to 2024-07-31) and checked on the gate (2024-08-01 to
2025-06-30). The parity check below confirms that the library functions
reproduce the saved research parquets (to 1e-12) over 2020-01-01 to
2025-06-30 before any lockbox data is used.

**The lockbox numbers are reported as they come out.** No construction,
weighting rule or lookback may change after this notebook runs. A code bug
(an exception, a parity failure, NaN output) may be fixed and the run
repeated; a modelling change may not.

Combination: walk-forward mean-variance (train_days=756, step_days=63,
min_train=252), the library defaults with no tuning, reported next to an
equal-weight baseline. Both rules are logged to the registry so the trial
count behind the deflated Sharpe stays complete.

In [1]:
from pathlib import Path

In [2]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

In [3]:
matplotlib.use("Agg")  # non-interactive; prevents display errors in nbconvert

In [4]:
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from quantlib import metrics, robustness, signals, strategies, trials

In [5]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
REGISTRY = PROC / "trial_registry.csv"

In [6]:
# Windows
DEV     = slice("2020-01-01", "2024-07-31")
GATE    = slice("2024-08-01", "2025-06-30")
LOCKBOX = slice("2025-07-01", None)        # opens here, once
FULL    = slice("2020-01-01", None)

In [7]:
# Full panels, lockbox included (no .loc[:GATE_END] here). The parity check
# below runs on data up to 2025-06-30 before anything later is used.
price    = pd.read_parquet(PROC / "price.parquet")
returns  = pd.read_parquet(PROC / "returns.parquet")
universe = pd.read_parquet(PROC / "universe.parquet")
funding  = pd.read_parquet(PROC / "funding.parquet")
taker    = pd.read_parquet(PROC / "taker_imbalance.parquet")

# Funding needs the universe's columns, as in notebook 04.
funding = funding.reindex(columns=universe.columns)

print("Panel extents (full, including lockbox):")
for name, df in [("price", price), ("returns", returns), ("universe", universe),
                 ("funding", funding), ("taker", taker)]:
    print(f"  {name}: {df.index.min().date()} → {df.index.max().date()}  shape={df.shape}")

Panel extents (full, including lockbox):
  price: 2018-01-01 → 2026-07-06  shape=(3109, 149)
  returns: 2018-01-01 → 2026-07-06  shape=(3109, 149)
  universe: 2018-01-01 → 2026-07-06  shape=(3109, 149)
  funding: 2019-09-10 → 2026-07-06  shape=(2492, 149)
  taker: 2018-01-01 → 2026-07-06  shape=(3109, 149)


In [8]:
# The three sleeves on the full panels, using the frozen defaults.
seasonality = strategies.seasonal_momentum_sleeve(price, returns, universe)
orderflow   = strategies.orderflow_sleeve(taker, returns, universe)
carry       = strategies.carry_sleeve(funding, returns, universe)

print("Sleeve extents (recomputed on full panels):")
for s in [seasonality, orderflow, carry]:
    r = s.dropna()
    print(f"  {s.name}: {r.index.min().date()} → {r.index.max().date()}  n={len(r)}")

Sleeve extents (recomputed on full panels):
  seasonality: 2018-02-01 → 2026-07-06  n=3078
  orderflow: 2018-01-02 → 2026-07-06  n=3108
  carry: 2019-09-10 → 2026-07-06  n=2492


In [9]:
# Parity check, before any lockbox data is read: each recomputed sleeve, cut
# to 2020-01-01:2025-06-30, must match its research parquet to within 1e-12.
# A failure means the frozen function no longer matches the research, and the
# run stops.
def _parity(recomputed: pd.Series, parquet_path: Path, col: str) -> None:
    saved = pd.read_parquet(parquet_path)[col]
    # Compare on the window the parquet covers
    r = recomputed.loc["2020-01-01":"2025-06-30"]
    s = saved.loc["2020-01-01":"2025-06-30"]
    check = pd.concat([r.rename("recomp"), s.rename("saved")], axis=1).dropna()
    if len(check) == 0:
        raise AssertionError(f"Parity check for {col}: no overlapping non-NaN rows")
    ok = np.allclose(check["recomp"].values, check["saved"].values, atol=1e-12)
    print(f"  {col}: {len(check)} rows compared, allclose={ok}")
    assert ok, (
        f"PARITY FAILURE for {col}: max abs diff = "
        f"{(check['recomp'] - check['saved']).abs().max():.2e}"
    )

print("Parity checks:")
_parity(seasonality, PROC / "sleeve_seasonality.parquet", "seasonality")
_parity(orderflow,   PROC / "sleeve_orderflow.parquet",   "orderflow")
_parity(carry,       PROC / "sleeve_carry.parquet",       "carry")
print("All parity checks PASSED.")

Parity checks:
  seasonality: 2008 rows compared, allclose=True
  orderflow: 2008 rows compared, allclose=True
  carry: 2008 rows compared, allclose=True
All parity checks PASSED.


## Which sleeves go in (decided before the lockbox)

These choices were made on the dev and gate windows before this notebook
ran.

**In: seasonality** (weekday-halved momentum, weekend=1.0, weekday=0.5).
It replaces the baseline momentum sleeve: the same trend book with a
calendar overlay that beat it on both dev and gate. Holding both would
double up the same position.

**In: orderflow** (taker follow, smooth=10, 7 bps). Positive on dev and
gate, with both neighbouring k values positive. The idea is that informed
flow predicts short-term direction.

**In: carry** (short high funding, k=7, 7 bps). Dev +1.01, gate +2.19, and
both neighbours (k=1, k=30) positive. Funding is carry the short side gets
paid, and it doubles as a crowding signal.

**Out: baseline pairs.** A dev Sharpe of -0.40 fails the dev > 0 rule.

**Out: fast reversal.** Every config is negative on both windows; costs eat
the gross edge (notebook 05).

In [10]:
# Sleeve correlations over the dev window.
sleeves_full = pd.concat([seasonality, orderflow, carry], axis=1).dropna()
print(f"Sleeves combined shape (full, dropna): {sleeves_full.shape}")
print(f"  Common index: {sleeves_full.index.min().date()} → {sleeves_full.index.max().date()}")

dev_corr = sleeves_full.loc[DEV].corr()
print("\nSleeve correlation matrix (DEV window 2020-01-01 → 2024-07-31):")
print(dev_corr.round(3))

Sleeves combined shape (full, dropna): (2492, 3)
  Common index: 2019-09-10 → 2026-07-06

Sleeve correlation matrix (DEV window 2020-01-01 → 2024-07-31):
             seasonality  orderflow  carry
seasonality        1.000      0.104 -0.083
orderflow          0.104      1.000  0.067
carry             -0.083      0.067  1.000


In [11]:
# n_search is the registry size before the portfolio rows go in: the 48
# research trials are the search space, while the two portfolio rows are
# combination rules rather than searched configs.
n_search = trials.n_trials(REGISTRY)
print(f"Registry rows before portfolio logging: {n_search}")
assert n_search == 48, f"Expected 48 research trial rows, got {n_search}"

Registry rows before portfolio logging: 48


In [12]:
# Drop portfolio rows left by an earlier run of this notebook, so a rerun does
# not count them twice. This happens before the two rows are logged below.
_reg = trials.load_registry(REGISTRY)
_pre_portfolio = _reg[_reg["family"] != "portfolio"]
if len(_pre_portfolio) < len(_reg):
    _trimmed = len(_reg) - len(_pre_portfolio)
    print(f"Trimming {_trimmed} prior portfolio row(s) from registry.")
    _pre_portfolio.to_csv(REGISTRY, index=False)
else:
    print("No prior portfolio rows found; registry clean.")

No prior portfolio rows found; registry clean.


In [13]:
# Combine the sleeves two ways: walk-forward mean-variance (library defaults,
# no tuning) and a plain equal-weight average.
wf, weight_path = robustness.walk_forward_weights(
    sleeves_full,
    train_days=756,
    step_days=63,
    min_train=252,
)
wf.name = "walk_forward"

ew = sleeves_full.mean(axis=1).rename("equal_weight")

print(f"Walk-forward: {wf.dropna().index.min().date()} → {wf.dropna().index.max().date()}  "
      f"n={len(wf.dropna())}")
print(f"Equal-weight: {ew.dropna().index.min().date()} → {ew.dropna().index.max().date()}  "
      f"n={len(ew.dropna())}")

Walk-forward: 2021-10-05 → 2026-07-06  n=1736
Equal-weight: 2019-09-10 → 2026-07-06  n=2492


In [14]:
# Log both combination rules (family "portfolio"). Each Sharpe uses whatever
# part of the window the series covers.
wf_dev_sharpe  = metrics.sharpe(wf.loc[DEV].dropna())
wf_gate_sharpe = metrics.sharpe(wf.loc[GATE].dropna())
ew_dev_sharpe  = metrics.sharpe(ew.loc[DEV].dropna())
ew_gate_sharpe = metrics.sharpe(ew.loc[GATE].dropna())

trials.log_trial(
    REGISTRY, "portfolio",
    {"rule": "walk_forward_mv", "train_days": 756, "step_days": 63, "min_train": 252},
    dev_sharpe=wf_dev_sharpe,
    gate_sharpe=wf_gate_sharpe,
    note="walk-forward mean-variance combination of 3 sleeves",
)
trials.log_trial(
    REGISTRY, "portfolio",
    {"rule": "equal_weight"},
    dev_sharpe=ew_dev_sharpe,
    gate_sharpe=ew_gate_sharpe,
    note="equal-weight naive baseline (3 sleeves)",
)

print(f"Logged walk_forward_mv: dev={wf_dev_sharpe:.3f}  gate={wf_gate_sharpe:.3f}")
print(f"Logged equal_weight:    dev={ew_dev_sharpe:.3f}  gate={ew_gate_sharpe:.3f}")
n_after = trials.n_trials(REGISTRY)
assert n_after == 50, f"Expected 50 registry rows after portfolio logging, got {n_after}"
print(f"Registry total: {n_after} rows (48 research + 2 portfolio). PASS.")

Logged walk_forward_mv: dev=0.397  gate=2.519
Logged equal_weight:    dev=1.681  gate=2.903
Registry total: 50 rows (48 research + 2 portfolio). PASS.


In [15]:
# Performance by window (metrics.summary) for both books and each sleeve.
windows = {
    "DEV":     DEV,
    "GATE":    GATE,
    "LOCKBOX": LOCKBOX,
    "FULL":    FULL,
}

series_map = {
    "walk_forward":  wf,
    "equal_weight":  ew,
    "seasonality":   seasonality,
    "orderflow":     orderflow,
    "carry":         carry,
}

rows = []
for win_name, win_slice in windows.items():
    for ser_name, ser in series_map.items():
        r = ser.loc[win_slice].dropna()
        if len(r) < 10:
            continue
        s = metrics.summary(r, n_trials=n_search, name=f"{ser_name}|{win_name}")
        s.name = (win_name, ser_name)
        rows.append(s)

table = pd.DataFrame(rows).T
table.columns = pd.MultiIndex.from_tuples(table.columns)
print("\n=== Performance table ===")
print(table.round(3).to_string())


=== Performance table ===
                         DEV                                                     GATE                                                 LOCKBOX                                                     FULL                                            
                walk_forward equal_weight seasonality orderflow    carry walk_forward equal_weight seasonality orderflow   carry walk_forward equal_weight seasonality orderflow    carry walk_forward equal_weight seasonality orderflow    carry
Ann. Return            0.029        0.250       0.184     0.206    0.299        0.216        0.400       0.128     0.244   0.883        0.114        0.162      -0.052     0.234    0.282        0.081        0.256       0.136     0.215    0.366
Ann. Vol               0.081        0.138       0.097     0.245    0.305        0.079        0.118       0.101     0.106   0.311        0.076        0.129       0.088     0.111    0.351        0.079        0.134       0.096     0.213    0.313
S

In [16]:
# HAC t-stat and block-bootstrap 95% CI for the walk-forward book.
print("\n=== Walk-forward significance ===")
for win_name, win_slice in windows.items():
    r = wf.loc[win_slice].dropna()
    if len(r) < 30:
        print(f"  {win_name}: insufficient data (n={len(r)})")
        continue
    hac_t = metrics.sharpe_tstat(r, hac=True)
    lo, hi = metrics.bootstrap_sharpe_ci(r, n_boot=2000, method="block", seed=42)
    sr = metrics.sharpe(r)
    print(f"  {win_name}: Sharpe={sr:.3f}  HAC t={hac_t:.3f}  "
          f"95% CI=[{lo:.3f}, {hi:.3f}]  n={len(r)}")


=== Walk-forward significance ===


  DEV: Sharpe=0.397  HAC t=0.684  95% CI=[-0.800, 1.545]  n=1031
  GATE: Sharpe=2.519  HAC t=2.350  95% CI=[0.356, 4.471]  n=334


  LOCKBOX: Sharpe=1.458  HAC t=1.500  95% CI=[-0.423, 3.444]  n=371


  FULL: Sharpe=1.020  HAC t=2.245  95% CI=[0.134, 1.904]  n=1736


In [17]:
# Deflated Sharpe of the walk-forward book, full sample and lockbox only,
# with n_trials = n_search = 48 (every trial before the portfolio step).
print("\n=== Deflated Sharpe (walk-forward, n_trials=48) ===")
for label, r in [("FULL",    wf.loc[FULL].dropna()),
                 ("LOCKBOX", wf.loc[LOCKBOX].dropna())]:
    if len(r) < 10:
        print(f"  {label}: insufficient data")
        continue
    dsr = metrics.deflated_sharpe(r, n_trials=n_search)
    sr  = metrics.sharpe(r)
    print(f"  {label}: Sharpe={sr:.3f}  DeflatedSharpe={dsr:.4f}  "
          f"(threshold: 0.95; n_trials={n_search}; n={len(r)})")


=== Deflated Sharpe (walk-forward, n_trials=48) ===
  FULL: Sharpe=1.020  DeflatedSharpe=0.4901  (threshold: 0.95; n_trials=48; n=1736)
  LOCKBOX: Sharpe=1.458  DeflatedSharpe=0.2049  (threshold: 0.95; n_trials=48; n=371)


In [18]:
# Alpha and beta of the walk-forward book against BTC and the equal-weight
# market, on the full sample and on the lockbox alone.
benchmarks = pd.read_parquet(PROC / "benchmarks.parquet")
btc = benchmarks["BTC"]
mkt = benchmarks["MKT"]

print("\n=== Alpha / Beta ===")
for label, r_slice in [("FULL", FULL), ("LOCKBOX", LOCKBOX)]:
    r = wf.loc[r_slice].dropna()
    ab = metrics.alpha_beta(r, btc, mkt, names=["BTC", "MKT"])
    if not ab:
        print(f"  {label}: no overlap with benchmarks")
        continue
    print(f"  {label} (n={len(pd.concat([r, btc, mkt], axis=1).dropna())}):")
    print(f"    alpha_ann={ab['alpha_ann']:.4f}  alpha_tstat={ab['alpha_tstat']:.3f}")
    print(f"    beta_BTC={ab['beta_BTC']:.3f}  beta_MKT={ab['beta_MKT']:.3f}")
    print(f"    R²={ab['r_squared']:.4f}  IR={ab['resid_info_ratio']:.3f}")


=== Alpha / Beta ===
  FULL (n=1736):
    alpha_ann=0.0830  alpha_tstat=2.325
    beta_BTC=-0.013  beta_MKT=-0.023
    R²=0.0819  IR=1.092
  LOCKBOX (n=371):
    alpha_ann=0.0948  alpha_tstat=1.253
    beta_BTC=-0.020  beta_MKT=-0.016
    R²=0.0603  IR=1.290


/var/folders/mp/kvy67lqd79qf63k1csl4jk_m0000gn/T/ipykernel_75354/1821068976.py:16: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  print(f"  {label} (n={len(pd.concat([r, btc, mkt], axis=1).dropna())}):")
/var/folders/mp/kvy67lqd79qf63k1csl4jk_m0000gn/T/ipykernel_75354/1821068976.py:16: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  print(f"  {label} (n={len(pd.concat([r, btc, mkt], axis=1).dropna())}):")


In [19]:
# Weight path of the walk-forward book.
fig, ax = plt.subplots(figsize=(12, 4))
for col in weight_path.columns:
    ax.plot(weight_path.index, weight_path[col], label=col, linewidth=0.8)
ax.axvline(pd.Timestamp("2025-07-01"), color="red", linestyle="--",
           linewidth=1.2, label="Lockbox open")
ax.set_title("Walk-forward sleeve weights (max-Sharpe)")
ax.set_ylabel("Weight")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "06_weight_path.png", dpi=120)
plt.close(fig)
print("Weight-path plot saved.")

Weight-path plot saved.


In [20]:
# Equity curves: walk-forward, equal weight and BTC.
fig, ax = plt.subplots(figsize=(12, 5))

start = "2020-01-01"
for ser, label, color in [
    (wf, "Walk-forward portfolio", "steelblue"),
    (ew, "Equal-weight portfolio", "darkorange"),
]:
    r = ser.loc[start:].dropna()
    eq = (1 + r).cumprod()
    ax.plot(eq.index, eq, label=label, color=color, linewidth=1.2)

# BTC normalised to 1 at start
btc_r = btc.loc[start:].dropna()
btc_eq = (1 + btc_r).cumprod()
ax.plot(btc_eq.index, btc_eq, label="BTC (normalised)", color="gray",
        linewidth=0.8, linestyle="--")

ax.axvline(pd.Timestamp("2024-08-01"), color="navy", linestyle=":",
           linewidth=1.0, label="Gate start")
ax.axvline(pd.Timestamp("2025-07-01"), color="red", linestyle="--",
           linewidth=1.2, label="Lockbox open")
ax.set_yscale("log")
ax.set_title("Equity curves (log scale), 2020-01-01 onward")
ax.set_ylabel("Cumulative return (1 = start)")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(ROOT / "reports" / "06_equity_curves.png", dpi=120)
plt.close(fig)
print("Equity curves plot saved.")

Equity curves plot saved.


In [21]:
# Save combined_portfolio.parquet (columns: walk_forward, equal_weight).
combined = pd.concat([wf, ew], axis=1)
combined.index.name = "Date"
combined.to_parquet(PROC / "combined_portfolio.parquet")
print(f"Saved combined_portfolio.parquet: shape={combined.shape}  "
      f"idx={combined.index.min().date()} → {combined.index.max().date()}")
print(combined.describe().round(4))

Saved combined_portfolio.parquet: shape=(2492, 2)  idx=2019-09-10 → 2026-07-06
       walk_forward  equal_weight
count     1736.0000     2492.0000
mean         0.0002        0.0006
std          0.0042        0.0069
min         -0.0198       -0.0412
25%         -0.0020       -0.0030
50%          0.0001        0.0002
75%          0.0022        0.0034
max          0.0220        0.1007


/var/folders/mp/kvy67lqd79qf63k1csl4jk_m0000gn/T/ipykernel_75354/3722440462.py:5: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat([wf, ew], axis=1)


In [22]:
# The last rows of the registry and the count per family.
print("\n=== Registry summary (final 10 rows) ===")
reg_final = trials.load_registry(REGISTRY)
print(reg_final.tail(10).to_string(index=False))
print(f"\nFamily counts:\n{reg_final['family'].value_counts().to_string()}")
print(f"\nTotal rows: {len(reg_final)}")


=== Registry summary (final 10 rows) ===
                timestamp    family                                                                            config  dev_sharpe  gate_sharpe                                                note
2026-07-07T07:28:05+00:00   fastrev                                                   {"lookback_h": 4, "rebal_h": 1}   -3.413990    -7.787095                                  hourly XS reversal
2026-07-07T07:28:05+00:00   fastrev                                                   {"lookback_h": 4, "rebal_h": 6}   -1.982773    -4.561274                                  hourly XS reversal
2026-07-07T07:28:06+00:00   fastrev                                                  {"lookback_h": 12, "rebal_h": 1}   -2.898354    -5.051563                                  hourly XS reversal
2026-07-07T07:28:06+00:00   fastrev                                                  {"lookback_h": 12, "rebal_h": 6}   -2.803072    -4.070703                                  ho

## Conclusion: the lockbox result

**Lockbox (2025-07-01 to 2026-07-06, 371 days, opened once).** Walk-forward
Sharpe **+1.458**, equal weight **+1.233**. Beta to BTC -0.020. Annualised
alpha 9.48% with a t-stat of 1.25, which is not significant on one year of
data. The block-bootstrap 95% CI, [-0.42, +3.44], includes zero.

**Deflated Sharpe against the 48 research trials: 0.490 on the full sample
and 0.205 on the lockbox.** Both are below the usual 0.95 bar, so the book
does not pass the significance test. Out-of-sample performance is positive
with almost no market exposure, but at this sample size and trial count it
cannot be told apart from selection luck.

**Sleeves on the lockbox.** Orderflow (+1.95) and carry (+0.88) held up.
Seasonality came in at -0.56: its gate outperformance did not last, which is
the regime risk flagged in notebook 02.

**Equal weight vs walk-forward.** Equal weight beats walk-forward on dev,
gate and the full sample (full: 1.76 vs 1.02) and loses only on the lockbox.
With three nearly uncorrelated sleeves, re-estimating mean-variance weights
adds little over a plain average.

**Summary.** A diversified, low-beta book with a positive out-of-sample
result that is not statistically proven. The trial registry and the
deflated Sharpe are what keep that claim in proportion.

**Added after the audit.** The seasonality sleeve's backtest never charged
for resizing the book between weekdays and weekends. Notebook 07 re-scores
this same book with those trades charged: walk-forward 1.490 and equal
weight 1.158 on the lockbox, deflated Sharpe 0.214. The numbers above are
left as they came out.

**Added later.** The block above that drops portfolio rows left by an
earlier run can never run: the 48-row check before it stops a rerun first.
The notebook is left as it ran; it is meant to run once. Two statements in
it are looser than they read. The parity check "to 1e-12" uses np.allclose,
which also allows a relative difference of 1e-5; the differences were in
fact zero, and notebook 08 repeats the check with no relative tolerance.
And Orderflow's k=10 had one tested neighbour, not two: k=5, positive on
both windows. The other tested value, k=1, lost on both (notebook 03).
"Opened once" holds for this protocol; the project's first version had run
the two baselines over most of the lockbox as twsq alphas (README).